<a href="https://colab.research.google.com/github/barkain/recsys-2026/blob/r54-second-gen-supervised-retriever/r54-second-gen-supervised-retriever-notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R54 Phase 3 Full — Colab T4 Runner

Trains R54 Phase 3 folds 1-4 on Colab GPU. Bring back per-fold `oof_lists.json` artifacts and run integration locally.

**Plan:** fold 1 first as a sanity check, then folds 2-4 if fold 1 looks good. Fold 0 is reused from the local Phase 3 smoke run.

**Reference:** `docs/r54_colab_runbook.md`

## Cell 1: GPU check + clone repo

In [ ]:
!nvidia-smi
!git clone https://github.com/barkain/recsys-2026.git
%cd recsys-2026
!git checkout r54-second-gen-supervised-retriever
!git log --oneline -3

## Cell 2: Install deps via uv

In [ ]:
!pip install -q uv
!uv sync
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
!uv run python -c "import torch; print('venv torch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

## Cell 3: Fetch R12 payload from GitHub Release

The R12 preprocessed dev payload (~114MB) is published as a release asset on the repo. No upload needed.

In [ ]:
import os, urllib.request
os.makedirs("exp/eval", exist_ok=True)
url = "https://github.com/barkain/recsys-2026/releases/download/r54-data/_R12_all_turns_payload.pkl"
dst = "exp/eval/_R12_all_turns_payload.pkl"
print(f"Downloading R12 payload from {url}...")
urllib.request.urlretrieve(url, dst)
print(f"Downloaded. Size: {os.path.getsize(dst):,} bytes")

**Option B (alternative):** mount Drive and copy from there.

```python
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/r54/_R12_all_turns_payload.pkl exp/eval/
```

## Cell 4: Pre-download HF datasets (one-time, ~2-3 min)

In [ ]:
from datasets import load_dataset
_ = load_dataset("talkpl-ai/TalkPlayData-Challenge-Track-Metadata")
_ = load_dataset("talkpl-ai/TalkPlayData-Challenge-Dataset")
print("HF datasets cached.")

### Add lightgbm to the venv (needed by some helper imports)

In [ ]:
!uv add lightgbm 2>&1 | tail -5
!uv run python -c "import lightgbm; print('lightgbm', lightgbm.__version__)"

## Cell 5: SANITY — fold 1 only

**Stop here and inspect output before continuing.** Expect:
- `Device: cuda`
- `fold 1: train_dev=6400  train_split=20000  total=26400`
- batches logging every 50, much faster than CPU (~2s/batch on T4)
- Final line: `fold 1 val hit@200: XXX/1600 (≥ 0.539)` — should be at or above Phase 2 fold-1 baseline.

Verified result (2026-05-15): hit@200 = 906/1600 = 0.566 (vs Phase 2 fold-1 0.539, +43).

In [ ]:
!uv run python scripts/expR54_phase3_full5fold_train.py --fold 1 --device cuda --no-aggregate

## Cell 6: Download fold 1 artifact (do this before running folds 2-4)

In [ ]:
!ls -lh cache/r54/phase3_full/fold_1/
!zip -r r54_phase3_fold1.zip cache/r54/phase3_full/fold_1/oof_lists.json
!ls -lh r54_phase3_fold1.zip
print("\nArtifact ready at /content/recsys-2026/r54_phase3_fold1.zip")
print("Grab it from the Colab file browser (left sidebar) or use one of:")
print("  - from google.colab import files; files.download('r54_phase3_fold1.zip')")
print("  - !cp r54_phase3_fold1.zip /content/drive/MyDrive/  (if Drive is mounted)")

## Cell 7: Folds 2, 3, 4 (only after fold 1 is verified)

In [ ]:
for fold_i in [2, 3, 4]:
    print(f"\n=== Running fold {fold_i} ===")
    !uv run python scripts/expR54_phase3_full5fold_train.py --fold {fold_i} --device cuda --no-aggregate

## Cell 8: Bundle all fold artifacts

Only `oof_lists.json` per fold is needed locally. The full `fold_*` dirs include model + embeddings (~600MB each), which are optional for Drive backup.

In [ ]:
import os, shutil
os.makedirs("/content/r54_phase3_artifacts", exist_ok=True)
for fold_i in [1, 2, 3, 4]:
    src = f"cache/r54/phase3_full/fold_{fold_i}/oof_lists.json"
    if os.path.exists(src):
        shutil.copy(src, f"/content/r54_phase3_artifacts/fold_{fold_i}_oof_lists.json")
        print(f"  fold {fold_i}: copied")
    else:
        print(f"  fold {fold_i}: MISSING")
!ls -la /content/r54_phase3_artifacts/
!cd /content && zip -r r54_phase3_artifacts.zip r54_phase3_artifacts/

## Cell 9: (optional) Full fold dirs incl. model + embeddings

In [ ]:
!du -sh cache/r54/phase3_full/fold_*
!zip -r /content/r54_phase3_full_folds1_4.zip \
    cache/r54/phase3_full/fold_1 \
    cache/r54/phase3_full/fold_2 \
    cache/r54/phase3_full/fold_3 \
    cache/r54/phase3_full/fold_4

In [ ]:
!unzip -t /content/r54_phase3_full_folds1_4.zip | tail -5
!ls -lh /content/r54_phase3_full_folds1_4.zip

## Cell 10: Drive backup (recommended before runtime ends)

Colab storage is temporary. If the session disconnects, artifacts are lost.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/r54_phase3_full_folds1_4.zip /content/drive/MyDrive/
!cp /content/r54_phase3_artifacts.zip /content/drive/MyDrive/ 2>/dev/null || true

## Local: install artifacts and evaluate

After downloading either zip locally:

```bash
cd /Users/nadavbarkai/dev/recsys-2026
unzip ~/Downloads/r54_phase3_artifacts.zip -d /tmp/r54_artifacts
for f in 1 2 3 4; do
  mkdir -p cache/r54/phase3_full/fold_$f
  cp /tmp/r54_artifacts/r54_phase3_artifacts/fold_${f}_oof_lists.json \
     cache/r54/phase3_full/fold_$f/oof_lists.json
done
mkdir -p cache/r54/phase3_full/fold_0
cp cache/r54/phase3_smoke/fold_0/oof_lists.json cache/r54/phase3_full/fold_0/oof_lists.json

# Aggregate into single oof_r54_lists.json
uv run python -c "
import sys; sys.path.insert(0, '.')
from scripts.expR54_phase3_full5fold_train import aggregate_oof_lists
from pathlib import Path
aggregate_oof_lists(Path('cache/r54/phase3_full'), 8000)
"

# Evaluate
uv run python scripts/expR54_phase3_full5fold_standalone.py
uv run python scripts/expR54_phase3_full5fold_integration.py
```